# 01 — EDA & Data Cleaning
**Lumen Data Science — Cross-Sell & Up-Sell Prediction**

Ovaj notebook pokriva:
1. Učitavanje i osnovni pregled podataka
2. Analizu stupaca — što zadržati, što baciti
3. Analizu nedostajućih vrijednosti
4. Čišćenje redova i outliera
5. Spremanje očišćenog dataseta

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

print('Libraries loaded ✓')

---
## 1. Učitavanje podataka


In [ ]:
import os

raw_path = '../data/raw/LUMEN_DS.csv'
converted_path = '../data/raw/data.csv'

if not os.path.exists(converted_path):
    with open(raw_path, 'r', encoding='utf-16') as file:
        data = file.read()
    
    data = data.replace('"', '')
    
    with open(converted_path, 'w', encoding='utf-8') as file:
        file.write(data)
    
    print('Konverzija gotova -> data.csv')
else:
    print('data.csv već postoji, preskačem konverziju')

In [ ]:
DATE_COLS = [
    'Invoice Date',
    'Order Date',
    'Customer First Invoice Date',
    'Price last modified date in the ERP',
    'Born on date'
]

df_raw = pd.read_csv(
    '../data/raw/data.csv',
    encoding='utf-8',
    sep='|',
    low_memory=False,
    parse_dates=DATE_COLS
)

print(f'Shape: {df_raw.shape}')
print(f'Redova: {df_raw.shape[0]:,} | Stupaca: {df_raw.shape[1]}')
df_raw.head(3)

In [ ]:
already_dropped = [
    'Invoice Line #', 'Order Date', 'Ordered qty',
    'Material cost of part', 'Labor cost of part', 'Overhead cost of part',
    'Sales Channel - Grouping', 'Born on date',
    'Price last modified date in the ERP',
    '# of unique products on a quote', 'Order Line #', 'Order #'
]

display(df_raw[already_dropped].head(20))

---
## 2. Osnovni pregled — dtypes, nullovi, unike

In [ ]:
# Pregled svakog stupca: dtype, % nullova, broj unikatnih vrijednosti
summary = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'null_count': df_raw.isnull().sum(),
    'null_%': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'n_unique': df_raw.nunique(),
    'sample_value': df_raw.iloc[0],
    'zero': df_raw.eq(0).sum(),
    'zero_%': (df_raw.eq(0).sum() / len(df_raw) * 100).round(2)
})
display(summary)

In [ ]:
# Vizualizacija % nullova po stupcu
null_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

if len(null_pct) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(null_pct) * 0.4)))
    null_pct.plot(kind='barh', ax=ax, color='salmon')
    ax.set_xlabel('% nedostajućih vrijednosti')
    ax.set_title('Postotak nullova po stupcu')
    ax.axvline(50, color='red', linestyle='--', alpha=0.7, label='50% granica')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Nema nullova — provjeri jesu li prazni stringovi umjesto NaN!')

In [ ]:
# Vizualizacija % zero-a po stupcu
zero_pct = (df_raw.eq(0).sum() / len(df_raw) * 100).sort_values(ascending=False)
zero_pct = zero_pct[zero_pct > 0]

if len(zero_pct) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(zero_pct) * 0.4)))
    zero_pct.plot(kind='barh', ax=ax, color='salmon')
    ax.set_xlabel('% vrijednosti nula')
    ax.set_title('Postotak zero-a po stupcu')
    ax.axvline(50, color='red', linestyle='--', alpha=0.7, label='50% granica')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Nema zero-a!')

In [ ]:
# Provjeri prazne stringove (čest problem u ovom datasetu)
empty_str = (df_raw == '').sum()
empty_str = empty_str[empty_str > 0]
print('Stupci s praznim stringovima:')
print(empty_str.sort_values(ascending=False))

---
## 3. Uklanjanje stupaca (nastavak od cleaner.py)

Prijateljev cleaner.py već je uklonio stupce koji su redudantni ili previše granularni.
Ovdje razmatramo treba li još nešto ukloniti.

In [ ]:
# Stupci koje je cleaner.py uklonio, ali malo editano
already_dropped = [
    'Invoice Line #', 'Order Date',
    'Material cost of part', 'Labor cost of part', 'Overhead cost of part',
    'Sales Channel - Grouping', 'Born on date',
    'Price last modified date in the ERP', 'Order Line #', 'Order #', '# of unique products on a quote'
]

# Primijeni drop (samo ako stupci postoje u df_raw)
cols_to_drop = [c for c in already_dropped if c in df_raw.columns]
df = df_raw.drop(columns=cols_to_drop)
print(f'Nakon inicijalnog dropa: {df.shape}')

In [ ]:
# --- ANALIZA PREOSTALIH STUPACA ---
# Pogledaj kardinalnost kategoričkih stupaca
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    print(f'\n=== {col} ({df[col].nunique()} unikatnih) ===')
    print(vc.head(10))

In [ ]:
# Vizualizacija % nullova po stupcu
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

if len(null_pct) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(null_pct) * 0.4)))
    null_pct.plot(kind='barh', ax=ax, color='salmon')
    ax.set_xlabel('% nedostajućih vrijednosti')
    ax.set_title('Postotak nullova po stupcu')
    ax.axvline(50, color='red', linestyle='--', alpha=0.7, label='50% granica')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Nema nullova — provjeri jesu li prazni stringovi umjesto NaN!')

In [ ]:
# Intercompany transakcije — treba li ih filtrirati?
# Intercompany = YES znači prodaja unutar iste korporacije, ne pravi kupac
print('Raspodjela Intercompany:')
print(df['Intercompany'].value_counts())
print(f'\n% Intercompany=YES: {(df["Intercompany"]=="YES").mean()*100:.1f}%')

# ODLUKA: Filtriraj intercompany van za cross-sell model
# (ostavi ih za posebnu analizu ako treba)
df_external = df[df['Intercompany'] == 'NO'].copy()
print(f'\nNakon filtriranja Intercompany=YES: {df_external.shape}')

In [ ]:
# Sales Channel - Internal vs External — jesu li isti?
same_channel = (df['Sales Channel - Internal'] == df['Sales Channel - External']).mean()
print(f'% redova gdje su Internal == External kanal: {same_channel*100:.1f}%')
# Ako je > 95%, jedan od njih je redundantan

# PRIJEDLOG: ako su gotovo uvijek isti, zadrži samo Internal i dropaj External
# df = df.drop(columns=['Sales Channel - External'])  # odkomentiraj ako potvrdiš

In [ ]:
display(df[['Sales Channel - Internal', 'Sales Channel - External']].head(20))

In [ ]:
# Manufacturing Location Code vs Manufacturing Region — hijerarhija?
print(df.groupby('Manufacturing Region')['Manufacturing Location Code'].nunique())
# Ako Region ima malo unikata a Location Code mnogo — Location Code je granularniji
# Za modele često je dovoljno Region razina

---
## 4. Čišćenje redova

In [ ]:
df_4 = df_external.copy()

print(df_4.columns.tolist())

In [ ]:
# Pregled svakog stupca: dtype, % nullova, broj unikatnih vrijednosti
summary_4 = pd.DataFrame({
    'dtype': df_4.dtypes,
    'null_count': df_4.isnull().sum(),
    'null_%': (df_4.isnull().sum() / len(df_4) * 100).round(2),
    'n_unique': df_4.nunique(),
    'sample_value': df_4.iloc[0],
    'zero': df_4.eq(0).sum(),
    'zero_%': (df_4.eq(0).sum() / len(df_4) * 100).round(2)
})
display(summary_4)

In [ ]:
# 4a. Ukloni redove bez ključnih identifikatora
before = len(df_4)
df_4 = df_4.dropna(subset=['Item Code', 'CustomerID'])
df_4 = df_4[df_4['CustomerID'] != -99]
print(f'Uklonjeno zbog null ključnih stupaca i customerID=-99: {before - len(df_4)} redova')

In [ ]:
bad_price = df_4['Invoiced price'].isna() | (df_4['Invoiced price'] == 0)
print(f'Ukupno redova bez prave cijene: {bad_price.sum():,}')
print(f'  - null: {df_4["Invoiced price"].isna().sum():,}')
print(f'  - zero: {(df_4["Invoiced price"] == 0).sum():,}')
print(f'  - negativne (zasebno rješavamo): {(df_4["Invoiced price"] < 0).sum():,}')
print(f'\nTo je {bad_price.mean()*100:.1f}% dataseta')
 

In [ ]:
sample_slash_mask = df_4['Item Code'].str.contains(r'SAMPLE|^/', na=False)
 
print('=== SAMPLE i slash (/) item codeovi — pregled ===\n')
print(f'Ukupno redova: {sample_slash_mask.sum():,}')
print(f'Unique item codeova: {df_4[sample_slash_mask]["Item Code"].nunique()}')
 
# Koliko ih ima pravu cijenu, koliko ne?
ss_with_price = (df_4[sample_slash_mask]['Invoiced price'] > 0).sum()
ss_without_price = (
    df_4[sample_slash_mask]['Invoiced price'].isna() | 
    (df_4[sample_slash_mask]['Invoiced price'] == 0)
).sum()
print(f'\nRedova s cijenom > 0: {ss_with_price:,} ({ss_with_price/sample_slash_mask.sum()*100:.1f}%)')
print(f'Redova s null/zero cijenom: {ss_without_price:,} ({ss_without_price/sample_slash_mask.sum()*100:.1f}%)')
 
# Breakdown po item codeu
ss_stats = (
    df_4[sample_slash_mask]
    .groupby('Item Code')
    .agg(
        total=('Invoiced price', 'size'),
        has_price=('Invoiced price', lambda x: (x > 0).sum())
    )
)
ss_stats['pct_with_price'] = (ss_stats['has_price'] / ss_stats['total'] * 100).round(1)
ss_stats = ss_stats.sort_values('total', ascending=False)
 
print('\nBreakdown po item codeu:')
display(ss_stats)

In [ ]:
before = len(df_4)
df_4 = df_4[~df_4['Item Code'].str.contains(r'SAMPLE|^/', na=False)]
print(f'Izbačeno SAMPLE/slash item codeova: {before - len(df_4)} redova')
print(f'Shape nakon: {df_4.shape}')
# izbacujemo item codeove koji pocinju s SAMPLE i /, oni ne bi trebali biti u finalnom datasetu po odgovoru Atomic Intelligence-a

In [ ]:
# Za SVAKI item code: koliki % svih redova ima pravu cijenu?
item_stats = df_4.groupby('Item Code').agg(
    total=('Invoiced price', 'size'),
    has_price=('Invoiced price', lambda x: (x > 0).sum())
)
item_stats['pct'] = item_stats['has_price'] / item_stats['total']
 
# Sensitivity — koliko redova spašavamo po pragu?
print('=== Sensitivity check — threshold vs. broj redova ===\n')
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    imputable = item_stats[item_stats['pct'] >= t].index
    n_saved = df_4[bad_price & df_4['Item Code'].isin(imputable)].shape[0]
    n_deleted = bad_price.sum() - n_saved
    n_items_saved = df_4.loc[bad_price & df_4['Item Code'].isin(imputable), 'Item Code'].nunique()
    n_items_deleted = df_4.loc[bad_price & ~df_4['Item Code'].isin(imputable), 'Item Code'].nunique()
    print(f'  Threshold {t:.0%}: impute {n_saved:>6,} redova ({n_items_saved:>3} itema) | '
          f'delete {n_deleted:>6,} redova ({n_items_deleted:>3} itema)')

In [ ]:
THRESHOLD = 0.5  # <-- prilagodi nakon sensitivity checka
# ili bi cak stavio 0.7
 
imputable_items = item_stats[item_stats['pct'] >= THRESHOLD].index
 
# Medijan SAMO od pozitivnih cijena
median_price = (
    df_4[df_4['Invoiced price'] > 0]
    .groupby('Item Code')['Invoiced price']
    .median()
)
 
# Impute gdje možemo
impute_mask = bad_price & df_4['Item Code'].isin(imputable_items)
df_4.loc[impute_mask, 'Invoiced price'] = (
    df_4.loc[impute_mask, 'Item Code'].map(median_price)
)
print(f'Imputirano: {impute_mask.sum():,} redova ({df_4.loc[impute_mask, "Item Code"].nunique()} itema)')
 
# Obriši gdje ne možemo
delete_mask = bad_price & ~df_4['Item Code'].isin(imputable_items)
print(f'Brisanje: {delete_mask.sum():,} redova ({df_4.loc[delete_mask, "Item Code"].nunique()} itema)')
print(f'\nTop 20 obrisanih item codeova:')
print(df_4[delete_mask]['Item Code'].value_counts().head(20))
 
before = len(df_4)
df_4 = df_4[~delete_mask]
print(f'\nShape: {before:,} → {len(df_4):,} (obrisano {before - len(df_4):,})')

In [ ]:
surviving_ss = df_4[df_4['Item Code'].str.contains(r'SAMPLE|^/', na=False)]
 
print('=== Sanity check — SAMPLE i slash item codeovi nakon imputacije ===\n')
print(f'Preživjelo redova: {len(surviving_ss):,}')
print(f'Unique item codeova: {surviving_ss["Item Code"].nunique()}')
 
if len(surviving_ss) > 0:
    ss_surv = (
        surviving_ss
        .groupby('Item Code')
        .agg(
            rows=('Invoiced price', 'size'),
            median_price=('Invoiced price', 'median'),
            min_price=('Invoiced price', 'min'),
            max_price=('Invoiced price', 'max')
        )
        .sort_values('rows', ascending=False)
    )
    print('\nPreživjeli SAMPLE/slash itemi:')
    display(ss_surv)
    print('\nAko ovo izgleda sumnjivo, razmisli o ručnom dropu.')
else:
    print('Nijedan SAMPLE/slash item nije preživio — čisto! ✓')
 
# Finalna provjera: ima li još null/zero cijena?
remaining = df_4['Invoiced price'].isna() | (df_4['Invoiced price'] == 0)
print(f'\nPreostalo redova s null/zero cijenom: {remaining.sum()}')

In [ ]:
# Za Invoiced price < 0
print(f'Redova s Invoiced price < 0: {(df_4["Invoiced price"] < 0).sum()}')
display(df_4[df_4['Invoiced price'] < 0].head(10))

In [ ]:
# Provjeri što je item code 46000
print(df_4[df_4['Item Code'] == '46000'][['Product family', 'Product group', 'Make vs Buy', 'Sales Channel - Internal']].value_counts().head(10))

# Ima li 46000 i normalnih (pozitivnih) transakcija?
print(f'\n46000 redova s cijenom > 0: {(df_4[df_4["Item Code"] == "46000"]["Invoiced price"] > 0).sum()}')
print(f'46000 redova s cijenom < 0: {(df_4[df_4["Item Code"] == "46000"]["Invoiced price"] < 0).sum()}')

# Kanal raspodjela negativnih
print('\nKanal raspodjela negativnih cijena:')
print(df_4[df_4['Invoiced price'] < 0]['Sales Channel - Internal'].value_counts().head(10))

In [ ]:
# Što su pozitivne 46000 transakcije
display(df_4[
    (df_4['Item Code'] == '46000') & 
    (df_4['Invoiced price'] > 0)
].head(5))

In [ ]:
before = len(df_4)
df_4 = df_4[df_4['Invoiced price'] >= 0]
print(f'Obrisano negativnih: {before - len(df_4)} redova')
print(f'Shape nakon: {df_4.shape}')

In [ ]:
print(f'Shape: {df_4.shape}')
print(f'\nNullovi:')
print(df_4.isnull().sum()[df_4.isnull().sum() > 0])
print(f'\nPrazni stringovi:')
empty = (df_4 == '').sum()
print(empty[empty > 0])

In [ ]:
print(f'Redova s Invoiced qty <= 0: {(df_4["Invoiced qty (shipped)"] <= 0).sum()}')
print(f'Redova s Invoiced qty < 0: {(df_4["Invoiced qty (shipped)"] < 0).sum()}')
print(f'Redova s Invoiced qty == 0: {(df_4["Invoiced qty (shipped)"] == 0).sum()}')

display(df_4[df_4['Invoiced qty (shipped)'] <= 0].head(5))

In [ ]:
print(f'Redova s Ordered qty <= 0: {(df_4["Ordered qty"] <= 0).sum()}')
print(f'Redova s Ordered qty < 0: {(df_4["Ordered qty"] < 0).sum()}')
print(f'Redova s Ordered qty == 0: {(df_4["Ordered qty"] == 0).sum()}')

display(df_4[df_4['Ordered qty'] <= 0].head(5))

In [ ]:
# Jesu li qty=0 redovi isporučeni ali bez zabilježene količine?
zero_qty = df_4[df_4['Invoiced qty (shipped)'] == 0]

# Imaju li normalnu cijenu?
print(f'% s cijenom > 0: {(zero_qty["Invoiced price"] > 0).mean()*100:.1f}%')

# Imaju li Ordered qty > 0?
print(f'% s Ordered qty > 0: {(zero_qty["Ordered qty"] > 0).mean()*100:.1f}%')

# Koliko jedinstvenih kupaca?
print(f'Unique kupaca: {zero_qty["CustomerID"].nunique()}')

# Koliko jedinstvenih item codeova?
print(f'Unique item codeova: {zero_qty["Item Code"].nunique()}')

In [ ]:
print(f'Negativni Invoiced qty: {(df_4["Invoiced qty (shipped)"] < 0).sum()}')
print(f'Negativni Ordered qty: {(df_4["Ordered qty"] < 0).sum()}')
print(f'Null Ordered qty: {df_4["Ordered qty"].isna().sum()}')

In [ ]:
# Kad je ordered qty == 0, kakav je invoiced qty?
print(df_4[df_4['Ordered qty'] == 0]['Invoiced qty (shipped)'].describe())
print()
print(df_4[df_4['Ordered qty'] == 0]['Invoiced qty (shipped)'].value_counts().head(10))

In [ ]:
# Makni samo negativne qty
before = len(df_4)
df_4 = df_4[df_4['Invoiced qty (shipped)'] >= 0]
df_4 = df_4[(df_4['Ordered qty'] >= 0) | (df_4['Ordered qty'].isna())]
print(f'Obrisano: {before - len(df_4)} redova')
print(f'Shape nakon: {df_4.shape}')

# Split na fakturirane i nefakturirane transakcije
df_invoiced_nula = df_4[df_4['Invoiced qty (shipped)'] == 0].copy()
df_4 = df_4[df_4['Invoiced qty (shipped)'] > 0].copy()

print(f"\ndf_invoiced (glavne transakcije): {len(df_4)}")
print(f"df_not_invoiced (intent za basket): {len(df_invoiced_nula)}")

In [ ]:
# Provjeri GM% distribuciju
print(df_4['GM%'].describe())
print(f'\nBroj null GM%: {df_4["GM%"].isna().sum()}')
print(f'\nBroj GM% > 1: {(df_4["GM%"] > 1).sum()}')
print(f'Broj GM% < -1: {(df_4["GM%"] < -1).sum()}')
print(f'Broj GM% < -5: {(df_4["GM%"] < -5).sum()}')
print(f'Broj GM% < -10: {(df_4["GM%"] < -10).sum()}')

import matplotlib.pyplot as plt
df_4['GM%'].clip(-2, 2).hist(bins=50)
plt.title('GM% distribucija (clipano na [-2, 2] za pregled)')
plt.show()

In [ ]:
# Pogledaj najgore outliere
print('Najmanji GM%:')
display(df_4.nsmallest(10, 'GM%')[['Item Code', 'CustomerID', 'Invoiced price', 'Cost of part', 'GM%']])

print('\nNajveći GM%:')
display(df_4.nlargest(10, 'GM%')[['Item Code', 'CustomerID', 'Invoiced price', 'Cost of part', 'GM%']])

In [ ]:
# Redovi di je cost drastično veći od cijene (npr. 100x+)
suspect = df_4[(df_4['Invoiced price'] > 0) & 
               (df_4['Cost of part'] > df_4['Invoiced price'] * 20)]
print(f"Suspect redova: {len(suspect)}")
print(suspect[['Item Code', 'CustomerID', 'Invoiced price', 'Cost of part', 'GM%']].head(20))

In [ ]:
# Izbaci garbage data (cost 20x+ veći od cijene)
mask_garbage = (df_4['Invoiced price'] > 0) & (df_4['Cost of part'] > df_4['Invoiced price'] * 20)
print(f"Garbage redova: {mask_garbage.sum()}")
df_4 = df_4[~mask_garbage]

In [ ]:
raw_path = '../data/raw/LUMEN_DS.csv'
converted_path1 = '../data/raw/data1.csv'

if not os.path.exists(converted_path1):
    with open(raw_path, 'r', encoding='utf-16') as file:
        data = file.read()
    
    data = data.replace('"', '')
    
    with open(converted_path1, 'w', encoding='utf-8') as file:
        file.write(data)
    
    print('Konverzija gotova -> data1.csv')
else:
    print('data1.csv već postoji, preskačem konverziju')

DATE_COLS = [
    'Invoice Date',
    'Order Date',
    'Customer First Invoice Date',
    'Price last modified date in the ERP',
    'Born on date'
]

df1 = pd.read_csv(
    '../data/raw/data1.csv',
    encoding='utf-8',
    sep='|',
    low_memory=False,
    parse_dates=DATE_COLS
)

print(df1.shape)

# Pogledaj najgore outliere na početnim podacima
print('Najmanji GM%:')
display(df1.nsmallest(10, 'GM%')[['Item Code', 'CustomerID', 'Invoiced price', 'Cost of part', 'GM%']])

print('\nNajveći GM%:')
display(df1.nlargest(10, 'GM%')[['Item Code', 'CustomerID', 'Invoiced price', 'Cost of part', 'GM%']])

In [ ]:
df_4['GM%'] = df_4['GM%'].clip(-5, 1)

print('Nakon winsorize:')
print(df_4['GM%'].describe())

In [ ]:
df_4.to_csv('../data/processed/data_clean_v1.csv', index=False)
print(f'Spremljeno! Shape: {df_4.shape}')

---
## 5. Popunjavanje nedostajućih vrijednosti

In [ ]:
DATE_COLS = [
    'Invoice Date',
    'Customer First Invoice Date'
]
df_5 = pd.read_csv('../data/processed/data_clean_v1.csv', low_memory=False, parse_dates=DATE_COLS)
print(f'Shape: {df_5.shape}')

In [ ]:
print(df_5.columns)
df_5.head(10)

In [ ]:
results = {}

for col in df_5.columns:
    issues = {}

    null_count = df_5[col].isna().sum()
    if null_count > 0:
        issues['NaN/null'] = null_count

    if df_5[col].dtype == 'object':
        empty_count = (df_5[col] == '').sum()
        if empty_count > 0:
            issues['empty string'] = empty_count

        unknown_count = (df_5[col].str.lower() == 'unknown').sum()
        if unknown_count > 0:
            issues['Unknown'] = unknown_count

        nan_str_count = (df_5[col].str.lower() == 'nan').sum()
        if nan_str_count > 0:
            issues['NaN string'] = nan_str_count

    if issues:
        results[col] = issues

for col, issues in results.items():
    total = sum(issues.values())
    print(f'\n{col} (ukupno: {total})')
    for issue_type, count in issues.items():
        print(f'{issue_type}: {count}')

In [ ]:
# Zamijeni prazne stringove s NaN za konzistentnost
df_5 = df_5.replace('', np.nan)

# Kategorički stupci — popuni s 'Unknown'
fill_unknown = [
    'Manufacturing Region',
    'Manufacturing Location Code',
    'Intercompany',
    'Customer industry',
    'Customer Region',
    'Top Customer Group',
    'Product family',
    'Product group',
    'Make vs Buy',
    'Sales Channel - Internal',
    'Sales Channel - External'
]

for col in fill_unknown:
    if col in df_5.columns:
        n_filled = df_5[col].isna().sum()
        df_5[col] = df_5[col].fillna('Unknown')
        if n_filled > 0:
            print(f'{col}: popunjeno {n_filled} redova s "Unknown"')

In [ ]:
results = {}

for col in df_5.columns:
    issues = {}

    null_count = df_5[col].isna().sum()
    if null_count > 0:
        issues['NaN/null'] = null_count

    if df_5[col].dtype == 'object':
        empty_count = (df_5[col] == '').sum()
        if empty_count > 0:
            issues['empty string'] = empty_count

        unknown_count = (df_5[col].str.lower() == 'unknown').sum()
        if unknown_count > 0:
            issues['Unknown'] = unknown_count

        nan_str_count = (df_5[col].str.lower() == 'nan').sum()
        if nan_str_count > 0:
            issues['NaN string'] = nan_str_count

    if issues:
        results[col] = issues

for col, issues in results.items():
    total = sum(issues.values())
    print(f'\n{col} (ukupno: {total})')
    for issue_type, count in issues.items():
        print(f'{issue_type}: {count}')

In [ ]:
"""n_before = df_5['Invoiced price (TX)'].isna().sum()
df_5['Invoiced price (TX)'] = df_5['Invoiced price (TX)'].fillna(df_5['Invoiced price'])
print(f'Invoiced price (TX): popunjeno {n_before} redova s Invoiced price')
"""

# ovo mi bas nema smisla raditi, svakako ni neznamo sta je taj TX i hocemo li koristiti

In [ ]:
n_before = df_5['Cost of part'].isna().sum()
print(f'Cost of part null: {n_before} redova')

In [ ]:
# Koliko Item Code-ova ima ISKLJUČIVO null Cost of part?
null_items = df_5.groupby('Item Code')['Cost of part'].apply(lambda x: x.isna().all())
print(f"Item Code-ovi gdje su SVI Cost of part null: {null_items.sum()}")
print(f"Ukupno redova u tim grupama: {df_5[df_5['Item Code'].isin(null_items[null_items].index)].shape[0]}")
# dakle svi koji imaju null cost of part nemaju nigdje drugdje neki cost of part

In [ ]:
"""median_cost_by_item = df_5.groupby('Item Code')['Cost of part'].median()
df_5['Cost of part'] = df_5['Cost of part'].fillna(df_5['Item Code'].map(median_cost_by_item))
print(f'Nakon Item Code: {df_5["Cost of part"].isna().sum()} nullova')"""

# ovo nema smisla jer su svi cost of partovi null 

In [ ]:
# Imaju li redovi bez Cost of part uopće smisleni GM%?
mask = df_5['Cost of part'].isna()
print(df_5.loc[mask, 'GM%'].describe())
print(df_5["Product group"].nunique())
print(df_5["Item Code"].nunique())

In [ ]:
"""avg_gm_by_group = df_5.groupby('Product group')['GM%'].median()
estimated = df_5['Invoiced price'] * (1 - df_5['Product group'].map(avg_gm_by_group))
df_5['Cost of part'] = df_5['Cost of part'].fillna(estimated)
print(f'Nakon GM% procjene po Product group: {df_5["Cost of part"].isna().sum()} nullova')"""

# mozemo se vratit na ovo ali preskacem za sad jer ima 32 product groups i 52k item codeova

In [ ]:
# Ćelija 1: Imputacija GM% gdje imamo Price i Cost
n_before = df_5['GM%'].isna().sum()

mask = df_5['GM%'].isna() & df_5['Cost of part'].notna() & (df_5['Invoiced price'] != 0)
df_5.loc[mask, 'GM%'] = (df_5.loc[mask, 'Invoiced price'] - df_5.loc[mask, 'Cost of part']) / df_5.loc[mask, 'Invoiced price']

n_after = df_5['GM%'].isna().sum()
print(f'GM% popunjeno: {n_before - n_after} redova')
print(f'GM% još null: {n_after}')

In [ ]:
# Ćelija 2: Provjera outliera
print(df_5['GM%'].describe())
print(f'\nGM% < -5: {(df_5["GM%"] < -5).sum()}')
print(f'GM% > 1: {(df_5["GM%"] > 1).sum()}')
print(f'GM% < -1: {(df_5["GM%"] < -1).sum()}')
print(f'GM% < 0: {(df_5["GM%"] < 0).sum()}')

In [ ]:
# Ćelija 3: Winsorize (nakon što vidiš rezultate ćelije 2, prilagodi granice)
df_5['GM%'] = df_5['GM%'].clip(-5, 1)
print(df_5['GM%'].describe())

In [ ]:
still_null_gm = df_5[df_5['GM%'].isna()]

print(f'Invoiced price null u tim redovima: {still_null_gm["Invoiced price"].isna().sum()}')
print(f'Cost of part null u tim redovima: {still_null_gm["Cost of part"].isna().sum()}')
print(f'Invoiced price == 0 u tim redovima: {(still_null_gm["Invoiced price"] == 0).sum()}')

In [ ]:
print(f'GM% null ostavljamo: {df_5["GM%"].isna().sum()}')
print(f'To je {df_5["GM%"].isna().sum() / len(df_5) * 100:.1f}% dataseta')

In [ ]:
# Spremi čist dataset
df_5.to_csv('../data/processed/data_clean_final.csv', index=False)
print(f'Spremljeno: {df_5.shape}')
print(f'Stupci: {df_5.columns.tolist()}')

---
## Sažetak — što smo napravili

| Korak | Akcija |
|-------|--------|
| Stupci | Uklonili smo 11 redudantnih stupaca (cleaner.py) |
| Intercompany | Filtrirali smo Intercompany=YES transakcije van modela |
| Redovi | Uklonili smo redove bez Item Code / CustomerID / Invoiced price |
| Redovi | Uklonili smo negativne cijene i količine (credit note-ovi) |
| GM% | Winsorized na [-5, 1] za outliere |
| Nullovi | Kategorički → 'Unknown', numerički → medijan |